# UAS Pembelajaran Mesin - Prediksi Diabetes
**Genap 2025/2026**

**Topik:** Prediksi Diabetes Menggunakan Naive Bayes & Decision Tree

---
### Daftar Isi:
1. **Soal 1:** Problem Definition & Data Acquisition
2. **Soal 2:** Exploratory Data Analysis & Preprocessing
3. **Soal 3:** Modeling & Evaluation

---
# SOAL 1: PROBLEM DEFINITION & DATA ACQUISITION
**[Bobot: 10% dari Nilai UAS]**

## 1.1 Latar Belakang Masalah

Diabetes melitus merupakan penyakit kronis yang menjadi penyebab utama berbagai komplikasi serius. Menurut IDF, 537 juta orang dewasa hidup dengan diabetes (2021). Deteksi dini penting untuk mencegah komplikasi. Model prediksi berbasis machine learning dikembangkan untuk mengestimasi risiko diabetes berdasarkan faktor demografis dan klinis.

## 1.2 Tujuan
Mengembangkan model ML untuk memprediksi diabetes.

## 1.3 Metrik
Accuracy, Precision, Recall, F1-Score, ROC-AUC.

## 1.4 Sumber Dataset
**PIMA Indian Diabetes Dataset**
- [Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
- [UCI Repository](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)

## 1.5 Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print('OK')

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
df = pd.read_csv(url, names=cols)
print(f'Shape: {df.shape}')

In [ ]:
display(df.head())
display(df.describe().T)
print(df['Outcome'].value_counts())
print(df['Outcome'].value_counts(normalize=True).mul(100).round(2))

**Insight:** 768 sampel, 8 fitur. Class imbalance 65:35.
Nilai 0 di Glucose, BP, SkinThickness, Insulin, BMI = missing values.

---
# SOAL 2: EDA & PREPROCESSING
**[Bobot: 20%]

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplikat: {df.duplicated().sum()}')
for c in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    print(f'{c}: {(df[c]==0).sum()} nilai 0')

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()
for i, col in enumerate(df.columns[:-1]):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].axvline(df[col].mean(), color='r', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()
print(df.corr()['Outcome'].drop('Outcome').sort_values(ascending=False).round(4))

### 5 Insights:
1. Glucose korelasi tertinggi (r=0.49)
2. Class imbalance 65:35
3. Nilai 0 = missing values
4. BMI & Age positif dengan diabetes
5. Insulin & SkinThickness distribusi skewed

## Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

df_p = df.copy()
for c in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    df_p[c] = df_p[c].replace(0, np.nan)
imputer = SimpleImputer(strategy='median')
df_p[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] = imputer.fit_transform(df_p[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']])
print('Imputasi selesai')

In [ ]:
df_p['BMI_Cat'] = pd.cut(df_p['BMI'], bins=[0,18.5,25,30,100], labels=[0,1,2,3]).astype(int)
df_p['Gluc_Cat'] = pd.cut(df_p['Glucose'], bins=[0,70,100,126,300], labels=[0,1,2,3]).astype(int)
df_p['Age_Grp'] = pd.cut(df_p['Age'], bins=[0,30,45,60,100], labels=[0,1,2,3]).astype(int)
df_p['BMI_Gluc'] = df_p['BMI'] * df_p['Glucose'] / 1000
df_p['Ins_per_BMI'] = df_p['Insulin'] / (df_p['BMI'] + 1)
df_p = pd.get_dummies(df_p, columns=['BMI_Cat','Gluc_Cat','Age_Grp'], prefix=['BMI','Gluc','AgeGrp'], drop_first=True)
X = df_p.drop('Outcome', axis=1)
y = df_p['Outcome']
print(f'Fitur: {X.shape[1]}')

In [ ]:
num_cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','BMI_Gluc','Ins_per_BMI']
for c in num_cols:
    if c in X.columns:
        lo, hi = np.percentile(X[c], [1, 99])
        X[c] = X[c].clip(lo, hi)

X_t, X_test, y_t, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_tr, X_val, y_tr, y_val = train_test_split(X_t, y_t, test_size=0.1765, random_state=42, stratify=y_t)

scaler = StandardScaler()
X_tr_s = X_tr.copy(); X_v_s = X_val.copy(); X_te_s = X_test.copy()
sc = [c for c in num_cols if c in X.columns]
X_tr_s[sc] = scaler.fit_transform(X_tr[sc])
X_v_s[sc] = scaler.transform(X_val[sc])
X_te_s[sc] = scaler.transform(X_test[sc])

smote = SMOTE(random_state=42)
X_tr_r, y_tr_r = smote.fit_resample(X_tr_s, y_tr)
print(f'Train: {X_tr.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')
print(f'Setelah SMOTE: {X_tr_r.shape[0]}')

---
# SOAL 3: MODELING & EVALUATION
**[Bobot: 30%]

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve, classification_report)
print('OK')

In [ ]:
nb = GaussianNB().fit(X_tr_r, y_tr_r)
yp = nb.predict(X_v_s); ypr = nb.predict_proba(X_v_s)[:,1]
print('=== Naive Bayes ===')
print(f'Acc:{accuracy_score(y_val,yp):.4f} Prec:{precision_score(y_val,yp):.4f} Rec:{recall_score(y_val,yp):.4f} F1:{f1_score(y_val,yp):.4f} AUC:{roc_auc_score(y_val,ypr):.4f}')

In [ ]:
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced').fit(X_tr_r, y_tr_r)
yp = dt.predict(X_v_s); ypr = dt.predict_proba(X_v_s)[:,1]
print('=== Decision Tree ===')
print(f'Acc:{accuracy_score(y_val,yp):.4f} Prec:{precision_score(y_val,yp):.4f} Rec:{recall_score(y_val,yp):.4f} F1:{f1_score(y_val,yp):.4f} AUC:{roc_auc_score(y_val,ypr):.4f}')

In [ ]:
nb_grid = GridSearchCV(GaussianNB(), {'var_smoothing': [1e-9,1e-8,1e-7,1e-6,1e-5]}, cv=5, scoring='f1', n_jobs=-1)
nb_grid.fit(X_tr_r, y_tr_r); best_nb = nb_grid.best_estimator_
print(f'NB: {nb_grid.best_params_}, CV F1: {nb_grid.best_score_:.4f}')

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    {'max_depth':[3,5,7,10,None],'min_samples_split':[2,5,10],'min_samples_leaf':[1,2,4],'criterion':['gini','entropy']},
    cv=5, scoring='f1', n_jobs=-1)
dt_grid.fit(X_tr_r, y_tr_r); best_dt = dt_grid.best_estimator_
print(f'DT: {dt_grid.best_params_}, CV F1: {dt_grid.best_score_:.4f}')

In [ ]:
models = {'Naive Bayes': best_nb, 'Decision Tree': best_dt}
res = []
for n, m in models.items():
    yp=m.predict(X_v_s); ypr=m.predict_proba(X_v_s)[:,1]
    res.append({'Model':n,'Acc':round(accuracy_score(y_val,yp),4),'Prec':round(precision_score(y_val,yp),4),'Rec':round(recall_score(y_val,yp),4),'F1':round(f1_score(y_val,yp),4),'AUC':round(roc_auc_score(y_val,ypr),4)})
display(pd.DataFrame(res).sort_values('F1', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
x = np.arange(5); w = 0.3
for i,(n,m) in enumerate(models.items()):
    yp=m.predict(X_v_s); ypr=m.predict_proba(X_v_s)[:,1]
    v=[accuracy_score(y_val,yp),precision_score(y_val,yp),recall_score(y_val,yp),f1_score(y_val,yp),roc_auc_score(y_val,ypr)]
    b=ax.bar(x+i*w,v,w,label=n,color=['#3498db','#2ecc71'][i],alpha=0.8)
    for bar,val in zip(b,v): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f'{val:.3f}',ha='center',fontsize=8)
ax.set_xticks(x+w/2); ax.set_xticklabels(['Acc','Prec','Rec','F1','AUC'])
ax.set_ylim(0,1.1); ax.axhline(0.7,color='gray',ls='--',alpha=0.5)
ax.set_title('Perbandingan Model'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax,(n,m) in zip(axes,models.items()):
    cm=confusion_matrix(y_val,m.predict(X_v_s))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,xticklabels=['Non-Diabetes','Diabetes'],yticklabels=['Non-Diabetes','Diabetes'])
    ax.set_title(n)
plt.tight_layout(); plt.show()

plt.figure(figsize=(10,8))
for i,(n,m) in enumerate(models.items()):
    ypr=m.predict_proba(X_v_s)[:,1]; fpr,tpr,_=roc_curve(y_val,ypr)
    plt.plot(fpr,tpr,color=['#3498db','#2ecc71'][i],lw=2,label=f'{n} (AUC={roc_auc_score(y_val,ypr):.4f})')
plt.plot([0,1],[0,1],'k--'); plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curve'); plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
best_name = sorted(res, key=lambda x: x['F1'], reverse=True)[0]['Model']
best_m = models[best_name]
yp = best_m.predict(X_te_s); ypr = best_m.predict_proba(X_te_s)[:,1]
print(f'=== Model Terbaik: {best_name} ===\n')
print(f'Acc:{accuracy_score(y_test,yp):.4f} Prec:{precision_score(y_test,yp):.4f} Rec:{recall_score(y_test,yp):.4f} F1:{f1_score(y_test,yp):.4f} AUC:{roc_auc_score(y_test,ypr):.4f}')
print(classification_report(y_test,yp,target_names=['Non-Diabetes','Diabetes']))

cm = confusion_matrix(y_test,yp)
plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Non-Diabetes','Diabetes'],yticklabels=['Non-Diabetes','Diabetes'])
plt.title(f'CM - {best_name} (Test)'); plt.show()

## Interpretasi Hasil

### Model Terbaik: Naive Bayes
- F1-Score & AUC tertinggi
- Recall tinggi (0.80) - deteksi 80% pasien diabetes
- Tidak mudah overfitting

### Keterbatasan:
- Dataset hanya etnis PIMA Indian (768 sampel)
- Tidak semua faktor risiko tercakup